
# 01Shapely: The Geometry Foundation

Every other library in this stackGeoPandas, PySAL, even the spatial parts of PostGIS you'll query
from Pythonis ultimately manipulating **Shapely geometry objects** underneath. If you understand
Shapely well, the rest of the stack stops feeling like magic and starts feeling like "oh, that's just
calling this Shapely method for me."

## The three geometry types you'll actually use

For land banking, you'll work almost entirely with these:

- **`Point`**a single coordinate. Used for: a building location, a geocoded address, a survey marker.
- **`LineString`**a connected sequence of points. Used for: a road, a boundary edge, a proposed
  infrastructure route.
- **`Polygon`**a closed shape with an outer boundary (and optionally holes). Used for: **a land
  parcel**this is the geometry type at the center of almost everything you'll build.

There are also `MultiPoint`, `MultiLineString`, and `MultiPolygon`the same three ideas, but representing
*multiple* disconnected shapes as one object (e.g. a land banking company's entire portfolio of
non-adjacent parcels as one `MultiPolygon`).


In [ ]:

from shapely.geometry import Point, LineString, Polygon

# A Point takes (x, y)for geographic data, that's (longitude, latitude), in that order.
# Buea, Cameroon is roughly at longitude 9.24, latitude 4.15
buea = Point(9.24, 4.15)
print(buea)
print("x (longitude):", buea.x)
print("y (latitude): ", buea.y)


In [ ]:

# A Polygon takes a list of (x, y) coordinate tuples that form a closed ring.
# You don't need to repeat the first point at the endShapely closes the ring for you.

# A small illustrative land parcel (made-up coordinates, roughly parcel-sized in a real projected system —
# more on why real lat/lon coordinates make area calculations misleading in notebook 03)
parcel_a = Polygon([
    (9.240, 4.150),
    (9.241, 4.150),
    (9.241, 4.151),
    (9.240, 4.151),
])

print(parcel_a)
print("Area (in coordinate-degree unitsnot yet meaningful, see notebook 03):", parcel_a.area)
print("Centroid:", parcel_a.centroid)
print("Bounds (minx, miny, maxx, maxy):", parcel_a.bounds)



## Plotting a geometry

Shapely objects have built-in rendering in Jupyterjust put one as the last line of a cell and it
draws itself. This is genuinely useful for sanity-checking a parcel shape while you're working.


In [ ]:

parcel_a  # just evaluating it in a Jupyter cell renders it



## Spatial predicatesthe questions you can ask about two geometries

This is the part that matters most for duplicate-sale detection. Shapely lets you ask precise
geometric questions about how two shapes relate to each other:

| Method | Question it answers |
|---|---|
| `.intersects(other)` | Do these two shapes share *any* space at all (even just touching)? |
| `.overlaps(other)` | Do they share *interior* space, without one fully containing the other? |
| `.contains(other)` | Is `other` entirely inside this shape? |
| `.within(other)` | Is this shape entirely inside `other`? (the reverse of `.contains`) |
| `.touches(other)` | Do they share a boundary but no interior space (e.g. adjacent parcels)? |
| `.disjoint(other)` | Do they share nothing at all? |
| `.equals(other)` | Are they the exact same shape? |

**For duplicate-sale detection specifically, `.overlaps()` and `.intersects()` are your main tools** —
if two parcels sold to different buyers overlap significantly, that's a red flag worth surfacing.


In [ ]:

# Two overlapping parcelssimulating a duplicate-sale scenario
parcel_b = Polygon([
    (9.2405, 4.1505),
    (9.2415, 4.1505),
    (9.2415, 4.1515),
    (9.2405, 4.1515),
])

print("Do parcel_a and parcel_b intersect?", parcel_a.intersects(parcel_b))
print("Do they overlap (share interior, neither fully contains the other)?", parcel_a.overlaps(parcel_b))

# The actual overlapping region, as its own geometryuseful to show a human reviewer exactly
# what portion of the two parcels is in dispute
overlap_region = parcel_a.intersection(parcel_b)
print("Overlap area (degree-units for now):", overlap_region.area)
overlap_region



## Geometric operationscreating new shapes from existing ones

| Method | What it does | A land banking use case |
|---|---|---|
| `.buffer(distance)` | Expands a shape outward by a distance | "Find all parcels within 200 units of this proposed road"buffer the road line, then check what intersects it |
| `.union(other)` | Combines two shapes into one | Merging adjacent parcels a company owns into one boundary for a subdivision plan |
| `.intersection(other)` | The overlapping region only | Exactly what you saw aboveshowing precisely which part of two parcels conflicts |
| `.difference(other)` | This shape, minus whatever overlaps with `other` | Showing the *non-disputed* portion of a parcel after subtracting a claimed overlap |
| `.convex_hull` | The smallest convex shape that contains it | Useful for cleaning up a messy, self-intersecting boundary from a bad digitization |


In [ ]:

# Buffering a proposed road (a LineString) to find nearby landthis is the core mechanic
# behind infrastructure-impact forecasting
road = LineString([(9.235, 4.148), (9.245, 4.152)])
road_buffer = road.buffer(0.003)  # expand outwardagain, degree-units for now, see notebook 03

print("Is parcel_a within range of the proposed road?", road_buffer.intersects(parcel_a))
road_buffer



## Exercises

Try each one yourself in the empty cell before checking the solution underneath. Use the `parcel_a`,
`parcel_b`, and `road` objects already defined above.

### Exercise 1
Create a third polygon, `parcel_c`, that does **not** overlap with either `parcel_a` or `parcel_b`.
Confirm this using `.intersects()` for both pairs.


In [ ]:
# Your code here


#### Solution

In [ ]:

parcel_c = Polygon([
    (9.250, 4.160),
    (9.251, 4.160),
    (9.251, 4.161),
    (9.250, 4.161),
])

print("parcel_c intersects parcel_a?", parcel_c.intersects(parcel_a))
print("parcel_c intersects parcel_b?", parcel_c.intersects(parcel_b))



### Exercise 2
Write a function `check_duplicate_sale(new_parcel, existing_parcels)` that takes a new parcel geometry
and a list of existing parcel geometries, and returns a list of the existing parcels it overlaps with.
This is literally the core logic of the duplicate-sale detection featureyou're building the real
thing, just without the database around it yet.


In [ ]:
# Your code here
# a parcel that will intersect with both parcel a and parcel c
parcel_d = Polygon([
    (9.2405, 4.1505),
    (9.251, 4.1505),
    (9.251, 4.161),
    (9.2405, 4.161)
])

#### Solution

In [ ]:

def check_duplicate_sale(new_parcel, existing_parcels):
    conflicts = []
    for existing in existing_parcels:
        if new_parcel.intersects(existing):
            conflicts.append(existing)
    return conflicts

existing = [parcel_a, parcel_c]
conflicts = check_duplicate_sale(parcel_d, existing)
print(f"parcel_d conflicts with {len(conflicts)} existing parcel(s)")



### Exercise 3
Using `.buffer()`, find whether `parcel_c` is within 0.02 degree-units of the `road` LineString, even
though it doesn't directly touch the road. (Don't worry about what "0.02 degree-units" means in real
distance yetthat's exactly the problem notebook 03 solves.)


In [ ]:
# Your code here
road_mini_buffer = road.buffer(0.001)  # smaller buffer to simulate a more precise impact zone
print("Is parcel_a within range of the proposed road (mini buffer)?", road_mini_buffer.intersects(parcel_a))
print("Is parcel_d within range of the proposed road (mini buffer)?", road_mini_buffer.intersects(parcel_d))
print("Is parcel_c within range of the proposed road (mini buffer)?", road_mini_buffer.intersects(parcel_c))
print("Is parcel_b within range of the proposed road (mini buffer)?", road_mini_buffer.intersects(parcel_b))
road_mini_buffer

#### Solution

In [ ]:

road_wide_buffer = road.buffer(0.02)
print("Is parcel_c within range?", road_wide_buffer.intersects(parcel_c))



## What's next

You now have the geometric vocabulary the rest of the stack builds on. The obvious limitation you
probably noticed: everything so far has been one shape at a time, in raw degree-based coordinates with
no real-world meaning, and nothing is connected to a table of actual parcel records (owner, price, date).
That's exactly what **`02_geopandas_fundamentals.ipynb`** solves next.
